# Usage Guidelines – Weather & Demand Merge Script

## 1. Required Files

Ensure the following files are available in the working directory:

* **Demand data file**

  * `cleaned_demand_data.csv`

* **Weather data file**

  * Example: `weather_TPSODL_GANJAM.csv`

---

## 2. Running the Merge Script

1. Open the script or notebook cell containing the merge code.
2. Update the file paths if necessary:

```
weather_file = "weather_TPSODL_GANJAM.csv"
demand_file = "cleaned_demand_data.csv"
```

3. Run the cell or script.

The script will:

* Load the weather and demand data
* Convert timestamps to datetime format
* Interpolate weather data from hourly to **15-minute frequency**
* Merge the datasets using the **Timestamp** column
* Round numeric values to **4 decimal places**

---

## 3. Output File

After execution, a merged dataset will be created:

```
merged_demand_weather_all_columns.csv
```

This file contains:

* All demand columns
* Weather variables
* A unified **15-minute timestamp**

---

## 4. Selecting Demand Zone and Weather Features

In the next step, you can choose:

* Any **demand column** (e.g., `TPCODL Demand`, `TPSOSDL Demand`, or `Total Demand (as recorded)`)
* Any subset of **weather variables**

Example selection:

```
selected_demand = "TPSOSDL Demand"

weather_features = [
    "temperature",
    "humidity",
    "precipitation",
    "wind_speed"
]
```

The resulting dataset will have the format:

```
Timestamp | demand | weather features
```

---

## 5. Notes

* Weather data is interpolated to match the **15-minute demand resolution**.
* The merge operation only needs to be run **once**.
* Different experiments can reuse the merged dataset by selecting different demand zones or weather features.


In [3]:
import pandas as pd

# -----------------------------
# USER INPUT
# -----------------------------
weather_file = r"C:\Users\91930\Desktop\SLDC complete\Notebooks\Weather\weather_by_zone\weather_TPCODL_ANGUL_5.csv"
demand_file = "cleaned_demand_data.csv"

# -----------------------------
# LOAD WEATHER
# -----------------------------
weather = pd.read_csv(weather_file)

weather["time"] = pd.to_datetime(weather["time"], errors="coerce")

# Rename weather columns
weather = weather.rename(columns={
    "temperature_2m (°C)": "temperature",
    "relative_humidity_2m (%)": "humidity",
    "rain (mm)": "precipitation",
    "wind_speed_10m (km/h)": "wind_speed"
})

weather = weather.sort_values("time")

# -----------------------------
# UPSAMPLE WEATHER → 15 MIN
# -----------------------------
weather = weather.set_index("time")

weather = weather.resample("15min").interpolate("linear")

weather = weather.reset_index().rename(columns={"time": "Timestamp"})

# -----------------------------
# LOAD DEMAND
# -----------------------------
demand = pd.read_csv(demand_file)

demand["Timestamp"] = pd.to_datetime(demand["Timestamp"], errors="coerce")

# -----------------------------
# MERGE
# -----------------------------
merged = pd.merge(
    demand,
    weather,
    on="Timestamp",
    how="inner"
)

merged = merged.round(4)

merged.to_csv("merged_demand_weather_all_columns.csv", index=False)

print("Merged dataset shape:", merged.shape)
print("Saved: merged_demand_weather_all_columns.csv")

C:\Users\91930\AppData\Local\Temp\ipykernel_22696\426427062.py:31: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  weather = weather.resample("15min").interpolate("linear")


Merged dataset shape: (318528, 21)
Saved: merged_demand_weather_all_columns.csv


In [5]:
import pandas as pd

df = pd.read_csv("merged_demand_weather_all_columns.csv")

# -----------------------------
# USER SELECTION
# -----------------------------
selected_demand = "Total Demand (as recorded)"

weather_features = [
    "temperature",
    "humidity",
    "precipitation",
    "wind_speed"
]

# -----------------------------
# CREATE FINAL DATASET
# -----------------------------
final_df = df[
    ["Timestamp", selected_demand] + weather_features
].copy()

final_df = final_df.rename(columns={
    selected_demand: "demand"
})

final_df = final_df.round(4)

print("Final dataset shape:", final_df.shape)

final_df.head()

Final dataset shape: (318528, 6)


,Timestamp,demand,temperature,humidity,precipitation,wind_speed
0,2017-01-01 00:00:00,2664.2819,20.400,92.0,0.0,5.70
1,2017-01-01 00:15:00,2698.8688,20.075,93.0,0.0,5.65
2,2017-01-01 00:30:00,2710.6829,19.750,94.0,0.0,5.60
3,2017-01-01 00:45:00,2696.4700,19.425,95.0,0.0,5.55
4,2017-01-01 01:00:00,2666.7276,19.100,96.0,0.0,5.50
